## Subset size trends with IQR

In [1]:
#!/usr/bin/env python3
"""
Subset size trend plots with IQR ribbons

Reads
  results/df_subsets.csv for subset level medians
  results/df_trees.csv for per tree distributions to compute IQR ribbons

Writes
  eval_addons/trend_subset_size_rf_iqr.(png|svg)
  eval_addons/trend_subset_size_bsd_iqr.(png|svg)
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_SUBS  = os.path.join("results", "df_subsets.csv")
IN_TREES = os.path.join("results", "df_trees.csv")
OUTDIR   = "eval_addons_trends_IQR"

GROUP_ORDER  = ["Amphibians", "Mammals", "Sharks", "Squamates"]
METHOD_ORDER = ["original", "at_root", "nearest_leaf_parent", "no_mcs", "supertree"]

METHOD_LABELS = {
    "original": "Proposed",
    "at_root": "Root Attach",
    "nearest_leaf_parent": "Nearest Parent",
    "no_mcs": "No MCS",
    "supertree": "Supertree-based",
}

def subset_to_size(subset_idx: int) -> int:
    return 50 + 5 * (subset_idx - 1)

def compute_iqr(df_trees: pd.DataFrame, metric_col: str) -> pd.DataFrame:
    """
    Returns a dataframe with columns
      group, subset, method_key, q25, q75
    """
    def q25(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return np.nan if x.size == 0 else float(np.quantile(x, 0.25))

    def q75(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return np.nan if x.size == 0 else float(np.quantile(x, 0.75))

    out = (
        df_trees
        .groupby(["group", "subset", "method_key"], as_index=False)[metric_col]
        .agg(q25=q25, q75=q75)
    )
    return out

def plot_trend_with_iqr(
    df_subsets: pd.DataFrame,
    df_iqr: pd.DataFrame,
    metric_median_col: str,
    metric_tree_col: str,
    ylabel: str,
    outbase: str
) -> None:
    groups = [g for g in GROUP_ORDER if g in df_subsets["group"].unique()]
    methods = [m for m in METHOD_ORDER if m in df_subsets["method_key"].unique()]

    fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, sharey=True)
    axes = axes.ravel()

    for ax, g in zip(axes, groups):
        dfg_med = df_subsets[df_subsets["group"] == g].copy()
        dfg_med["size"] = dfg_med["subset"].astype(int).apply(subset_to_size)

        dfg_iqr = df_iqr[df_iqr["group"] == g].copy()
        dfg_iqr["size"] = dfg_iqr["subset"].astype(int).apply(subset_to_size)

        for m in methods:
            med = dfg_med[dfg_med["method_key"] == m].sort_values("size")
            if med.empty:
                continue

            line = ax.plot(
                med["size"].values,
                med[metric_median_col].values,
                marker="o",
                linewidth=1.7,
                markersize=3.8,
                label=METHOD_LABELS.get(m, m),
            )[0]

            band = dfg_iqr[dfg_iqr["method_key"] == m].sort_values("size")
            if not band.empty:
                ax.fill_between(
                    band["size"].values,
                    band["q25"].values,
                    band["q75"].values,
                    alpha=0.15,
                    color=line.get_color(),
                    linewidth=0,
                )

        ax.set_title(g)
        ax.set_xlabel("Number of taxa in subset")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)
        ax.set_xticks([50, 80, 110, 145])

    for j in range(len(groups), len(axes)):
        axes[j].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=min(len(labels), 5),
        frameon=False,
        bbox_to_anchor=(0.5, -0.02),
    )

    fig.suptitle(f"Trend vs subset size with IQR ribbons: {ylabel}", y=1.02)
    fig.tight_layout()

    os.makedirs(OUTDIR, exist_ok=True)
    png = os.path.join(OUTDIR, f"{outbase}.png")
    svg = os.path.join(OUTDIR, f"{outbase}.svg")
    fig.savefig(png, dpi=200, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)

    print("Saved", png)
    print("Saved", svg)

def main():
    if not os.path.exists(IN_SUBS):
        raise FileNotFoundError(f"Missing {IN_SUBS}")
    if not os.path.exists(IN_TREES):
        raise FileNotFoundError(f"Missing {IN_TREES}")

    df_subsets = pd.read_csv(IN_SUBS)
    df_trees = pd.read_csv(IN_TREES)

    needed_subs = {"group", "subset", "method_key", "median_normalized_rf", "median_bsd"}
    needed_trees = {"group", "subset", "method_key", "tree_idx", "normalized_rf", "bsd"}

    missing_subs = needed_subs - set(df_subsets.columns)
    missing_trees = needed_trees - set(df_trees.columns)

    if missing_subs:
        raise ValueError(f"df_subsets.csv missing columns {missing_subs}")
    if missing_trees:
        raise ValueError(f"df_trees.csv missing columns {missing_trees}")

    iqr_rf = compute_iqr(df_trees, "normalized_rf")
    iqr_bsd = compute_iqr(df_trees, "bsd")

    plot_trend_with_iqr(
        df_subsets=df_subsets,
        df_iqr=iqr_rf,
        metric_median_col="median_normalized_rf",
        metric_tree_col="normalized_rf",
        ylabel="Median normalized RF",
        outbase="trend_subset_size_rf_iqr",
    )

    plot_trend_with_iqr(
        df_subsets=df_subsets,
        df_iqr=iqr_bsd,
        metric_median_col="median_bsd",
        metric_tree_col="bsd",
        ylabel="Median normalized BSD",
        outbase="trend_subset_size_bsd_iqr",
    )

if __name__ == "__main__":
    main()

Saved eval_addons_trends_IQR\trend_subset_size_rf_iqr.png
Saved eval_addons_trends_IQR\trend_subset_size_rf_iqr.svg
Saved eval_addons_trends_IQR\trend_subset_size_bsd_iqr.png
Saved eval_addons_trends_IQR\trend_subset_size_bsd_iqr.svg
